# 5 — Descriptive Analysis and Visualization

Implements Stage 5 from `research_project_plan_v2.md`.

**Inputs:**
- `data/chunk_features.parquet` or `data/chunk_features.csv`
- `data/session_features.parquet` or `data/session_features.csv`
- `data/chunk_registry_v1.csv`
- optional validation outputs under `data/validation/`

**Outputs:**
- `figures/conference_outcome_distributions.png`
- `figures/feature_conference_heatmap.png`
- `figures/chunk_position_profiles.png`
- `figures/irr_agreement_barplot.png` (if validation files exist)
- `figures/feature_distributions/<feature_name>_hist.png`
- `figures/feature_distribution_summary.csv`
- `figures/session_outlier_flags.csv`


## Step 0 — Imports and paths

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

NOTEBOOK_DIR = Path().resolve()
ANALYSIS_V2 = NOTEBOOK_DIR.parent
DATA_DIR = ANALYSIS_V2 / 'data'
FIG_DIR = ANALYSIS_V2 / 'figures'
FEATURE_DIST_DIR = FIG_DIR / 'feature_distributions'
VALIDATION_DIR = DATA_DIR / 'validation'

FIG_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIST_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR         :', DATA_DIR)
print('FIG_DIR          :', FIG_DIR)
print('VALIDATION_DIR   :', VALIDATION_DIR)


## Step 1 — Load Stage 4 outputs

In [ ]:
def load_table(stem: str) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f'Falling back to CSV for {stem}: {type(e).__name__}: {e}')
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path.name} or {csv_path.name}')


chunk_df = load_table('chunk_features')
session_df = load_table('session_features')
registry = pd.read_csv(DATA_DIR / 'chunk_registry_v1.csv')

session_meta = (
    registry.groupby('session_group', as_index=False)
    .agg(
        conference_id=('conference_id', 'first'),
        num_teams=('num_teams', 'max'),
        num_funded_teams=('num_funded_teams', 'max'),
        has_teams=('has_teams', 'max'),
        has_funded_teams=('has_funded_teams', 'max'),
        outcome_has_funded_teams=('outcome_has_funded_teams', 'max'),
        n_chunks=('chunk_id', 'count')
    )
)

session_plot_df = session_df.merge(session_meta, on='session_group', how='left')

print(f'chunk_df        : {chunk_df.shape[0]:,} rows x {chunk_df.shape[1]} cols')
print(f'session_df      : {session_df.shape[0]:,} rows x {session_df.shape[1]} cols')
print(f'session_plot_df : {session_plot_df.shape[0]:,} rows x {session_plot_df.shape[1]} cols')

display(session_plot_df[['session_group', 'conference_id', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams']].head())


## Step 2 — Conference-level summaries

Stage 5.1 asks for outcome distributions by conference, plus conference-level feature summaries and a feature/outcome heatmap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

for ax, outcome_col, title in [
    (axes[0], 'num_teams', 'Teams formed by conference'),
    (axes[1], 'num_funded_teams', 'Funded teams by conference'),
]:
    sns.violinplot(data=session_plot_df, x='conference_id', y=outcome_col, inner=None, cut=0, ax=ax, color='#9ecae1')
    sns.stripplot(data=session_plot_df, x='conference_id', y=outcome_col, ax=ax, color='black', alpha=0.55, size=4)
    ax.set_title(title)
    ax.set_xlabel('Conference')
    ax.set_ylabel(outcome_col)
    ax.tick_params(axis='x', rotation=45)

conference_outcome_path = FIG_DIR / 'conference_outcome_distributions.png'
fig.savefig(conference_outcome_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', conference_outcome_path)

conference_key_features = [
    'session_mean_collective_engagement_score',
    'session_mean_gini_coefficient',
    'session_mean_commitment_signal',
    'session_mean_cross_disciplinary_bridging',
    'session_mean_pct_turns_audible_backchannel',
    'session_mean_mean_nod_rate',
    'session_mean_mean_vocal_enthusiasm',
]
conference_key_features = [c for c in conference_key_features if c in session_plot_df.columns]

conference_summary = (
    session_plot_df.groupby('conference_id')[conference_key_features]
    .agg(['mean', 'std'])
    .round(3)
)
display(conference_summary)

heatmap_means = (
    session_plot_df.groupby('conference_id')[conference_key_features]
    .mean()
)

corr_features = conference_key_features + ['num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams']
corr_features = [c for c in corr_features if c in session_plot_df.columns]
corr_df = session_plot_df[corr_features].apply(pd.to_numeric, errors='coerce').corr()
outcome_corr = corr_df.loc[conference_key_features, [c for c in ['num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams'] if c in corr_df.columns]]

fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
sns.heatmap(heatmap_means.T, cmap='crest', center=heatmap_means.values.mean(), ax=axes[0])
axes[0].set_title('Conference mean feature levels')
axes[0].set_xlabel('Conference')
axes[0].set_ylabel('Feature')

sns.heatmap(outcome_corr, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Session feature correlations with outcomes')
axes[1].set_xlabel('Outcome')
axes[1].set_ylabel('Feature')

heatmap_path = FIG_DIR / 'feature_conference_heatmap.png'
fig.savefig(heatmap_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', heatmap_path)


## Step 3 — Chunk-position analysis

Stage 5.2 asks for chunk-position profiles overall and by whether sessions formed teams.

In [ ]:
chunk_order = ['beginning', 'middle', 'end', 'whole']
plot_chunk_features = [
    'collective_engagement_score',
    'gini_coefficient',
    'is_convergent',
    'commitment_signal',
]
plot_chunk_features = [c for c in plot_chunk_features if c in chunk_df.columns]

chunk_plot_df = chunk_df.copy()
chunk_plot_df['chunk_position'] = pd.Categorical(chunk_plot_df['chunk_position'], categories=chunk_order, ordered=True)
chunk_plot_df = chunk_plot_df[chunk_plot_df['chunk_position'].notna()].copy()
chunk_plot_df['team_formation_label'] = np.where(chunk_plot_df['has_teams'] == 1, 'Formed team', 'No team formed')

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
axes = axes.flatten()

for ax, feature in zip(axes, plot_chunk_features):
    summary = (
        chunk_plot_df.groupby(['chunk_position', 'team_formation_label'], observed=False)[feature]
        .mean()
        .reset_index()
    )
    sns.lineplot(data=summary, x='chunk_position', y=feature, hue='team_formation_label', marker='o', ax=ax)
    ax.set_title(feature)
    ax.set_xlabel('Chunk position')
    ax.set_ylabel('Mean value')
    ax.legend(title='Outcome')

for ax in axes[len(plot_chunk_features):]:
    ax.axis('off')

chunk_profile_path = FIG_DIR / 'chunk_position_profiles.png'
fig.savefig(chunk_profile_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', chunk_profile_path)


## Step 4 — Validation results visualization

If Stage 3 outputs are present, generate the human IRR and human-AI agreement bar plot described in Stage 5.3.

In [ ]:
irr_path = VALIDATION_DIR / 'human_irr_results.csv'
hai_path = VALIDATION_DIR / 'human_ai_agreement.csv'

def reliability_band(v):
    if pd.isna(v):
        return 'missing'
    if v >= 0.60:
        return 'high'
    if v >= 0.40:
        return 'moderate'
    return 'low'


palette = {
    'high': '#2e8b57',
    'moderate': '#d9a441',
    'low': '#c44e52',
    'missing': '#bdbdbd',
}

if irr_path.exists() and hai_path.exists():
    irr_df = pd.read_csv(irr_path)
    hai_df = pd.read_csv(hai_path)

    irr_kappa = irr_df[irr_df['metric'].astype(str).str.contains('kappa', case=False, na=False)].copy()
    hai_kappa = hai_df[hai_df['metric'].astype(str).str.contains('kappa', case=False, na=False)].copy()

    irr_kappa = irr_kappa[['field', 'value']].rename(columns={'value': 'human_irr_kappa'})
    hai_kappa = hai_kappa[['field', 'value']].rename(columns={'value': 'human_ai_kappa'})
    reliability_df = irr_kappa.merge(hai_kappa, on='field', how='outer').sort_values('field')

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(reliability_df) * 0.35)), constrained_layout=True)

    left_colors = reliability_df['human_irr_kappa'].map(reliability_band).map(palette)
    right_colors = reliability_df['human_ai_kappa'].map(reliability_band).map(palette)

    axes[0].barh(reliability_df['field'], reliability_df['human_irr_kappa'], color=left_colors)
    axes[0].axvline(0.60, color='gray', linestyle='--', linewidth=1)
    axes[0].axvline(0.40, color='gray', linestyle=':', linewidth=1)
    axes[0].set_title('Human IRR kappa')
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel('Kappa')

    axes[1].barh(reliability_df['field'], reliability_df['human_ai_kappa'], color=right_colors)
    axes[1].axvline(0.60, color='gray', linestyle='--', linewidth=1)
    axes[1].axvline(0.40, color='gray', linestyle=':', linewidth=1)
    axes[1].set_title('Human-AI agreement kappa')
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Kappa')

    irr_plot_path = FIG_DIR / 'irr_agreement_barplot.png'
    fig.savefig(irr_plot_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', irr_plot_path)
    display(reliability_df)
else:
    print('Validation files not found. Skipping Stage 5.3 plot.')
    print('Expected:', irr_path)
    print('Expected:', hai_path)


## Step 5 — Feature distributions and outlier flagging

Stage 5.4 asks for distribution checks and potential outlier identification.

In [ ]:
EXCLUDE_DIST = {
    'session_group', 'conference_id', 'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams',
    'outcome_has_funded_teams', 'n_chunks'
}

dist_features = [
    c for c in session_plot_df.columns
    if c not in EXCLUDE_DIST and pd.api.types.is_numeric_dtype(session_plot_df[c])
]

summary_rows = []
outlier_rows = []

for feature in tqdm(dist_features, desc='Feature distributions', unit='feature'):
    vals = pd.to_numeric(session_plot_df[feature], errors='coerce').dropna()
    if vals.empty:
        continue

    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    skew = vals.skew()
    suggested_transform = 'log_or_winsorize' if abs(skew) > 1 else 'none'

    summary_rows.append({
        'feature': feature,
        'n_non_missing': len(vals),
        'mean': vals.mean(),
        'std': vals.std(),
        'min': vals.min(),
        'max': vals.max(),
        'skew': skew,
        'iqr_lower_bound': lower,
        'iqr_upper_bound': upper,
        'suggested_transform': suggested_transform,
    })

    outlier_mask = (pd.to_numeric(session_plot_df[feature], errors='coerce') < lower) | (pd.to_numeric(session_plot_df[feature], errors='coerce') > upper)
    flagged = session_plot_df.loc[outlier_mask, ['session_group', 'conference_id']].copy()
    flagged['feature'] = feature
    flagged['value'] = pd.to_numeric(session_plot_df.loc[outlier_mask, feature], errors='coerce').values
    outlier_rows.append(flagged)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(vals, bins=25, kde=True, ax=ax, color='#4c72b0')
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    fig.savefig(FEATURE_DIST_DIR / f'{feature}_hist.png', dpi=160, bbox_inches='tight')
    plt.close(fig)

feature_summary_df = pd.DataFrame(summary_rows).sort_values(['suggested_transform', 'skew'], ascending=[False, False])
feature_summary_path = FIG_DIR / 'feature_distribution_summary.csv'
feature_summary_df.to_csv(feature_summary_path, index=False)

if outlier_rows:
    outlier_df = pd.concat(outlier_rows, ignore_index=True)
    outlier_df = outlier_df.sort_values(['feature', 'conference_id', 'session_group'])
else:
    outlier_df = pd.DataFrame(columns=['session_group', 'conference_id', 'feature', 'value'])

outlier_path = FIG_DIR / 'session_outlier_flags.csv'
outlier_df.to_csv(outlier_path, index=False)

print(f'Saved {len(feature_summary_df):,} feature summaries -> {feature_summary_path}')
print(f'Saved {len(outlier_df):,} outlier flags -> {outlier_path}')
print(f'Saved histogram files -> {FEATURE_DIST_DIR}')

display(feature_summary_df.head(15))
display(outlier_df.head(15))
